# Fase 5.1 — Preparacion del dataset WikiArt

Construye el corpus de 500 imagenes (50 x 10 estilos) para el fine-tuning LoRA.
Usa **modo streaming**: solo se descargan las imagenes necesarias (~200-400 MB),
no los 8 GB completos del dataset.

**Ejecutar antes de:** `fase5_finetune.ipynb`.
**Prerequisito:** `TFM_setup.ipynb` (Drive montado).

---
## 0 — Entorno

In [ ]:
from google.colab import drive
import os, sys, json

drive.mount("/content/drive", force_remount=False)
PROJECT_ROOT = "/content/drive/MyDrive/TFM/repositorio"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["HF_HOME"] = "/content/.cache/huggingface"
print(f"Proyecto: {PROJECT_ROOT}")

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = ""  # <-- pega tu token hf_...

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN vacio. Rellena la variable con tu token de "
        "https://huggingface.co/settings/tokens (tipo Read) "
        "y vuelve a ejecutar esta celda."
    )

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print("Autenticado en Hugging Face.")

In [ ]:
!pip install -q datasets
import datasets as _hf
print(f"datasets={_hf.__version__}")

---
## 1 — Configuracion

In [ ]:
# Parámetros del corpus WikiArt. Se seleccionan 10 estilos artísticos que
# cubren el espectro histórico desde el Barroco hasta el Art Nouveau, con
# especial representación de movimientos con tratamientos lumínicos y cromáticos
# característicos (impresionismo, expresionismo, barroco) para maximizar la
# señal detectable en las métricas de deriva semántica de la Fase 5.
# 50 imágenes por estilo (500 total) es un compromiso entre cobertura estadística
# y tiempo de fine-tuning viable en la GPU T4 (≤15 GB VRAM).
N_PER_STYLE = 50
IMG_SIZE    = 512
RANDOM_SEED = 42
BUFFER_SIZE = 2000   # buffer de shuffle en streaming

FINETUNE_DIR = os.path.join(PROJECT_ROOT, "data", "finetune")
IMAGES_DIR   = os.path.join(FINETUNE_DIR, "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

STYLES = [
    "Impressionism", "Realism", "Romanticism", "Expressionism",
    "Post_Impressionism", "Baroque", "Cubism", "Abstract_Art",
    "Symbolism", "Art_Nouveau_(Modern)",
]

STYLE_ADJ = {
    "Impressionism":        "impressionist",
    "Realism":              "realist",
    "Romanticism":          "romantic",
    "Expressionism":        "expressionist",
    "Post_Impressionism":   "post-impressionist",
    "Baroque":              "baroque",
    "Cubism":               "cubist",
    "Abstract_Art":         "abstract",
    "Symbolism":            "symbolist",
    "Art_Nouveau_(Modern)": "art nouveau",
}

print(f"Estilos  : {len(STYLES)}")
print(f"Imagenes : {N_PER_STYLE} x {len(STYLES)} = {N_PER_STYLE * len(STYLES)}")
print(f"Salida   : {FINETUNE_DIR}")

---
## 2 — Inicializacion del dataset en modo streaming

In [ ]:
# Carga del dataset WikiArt en modo streaming para evitar descargar los 8 GB
# completos. El modo streaming itera directamente sobre el servidor de HuggingFace,
# descargando solo los ejemplos que se van a usar (~200-400 MB en total).
# El shuffle con buffer_size=2000 garantiza diversidad dentro de cada estilo
# sin necesitar el dataset completo en memoria.
from datasets import load_dataset

print("Inicializando huggan/wikiart en modo streaming...")
print("(Solo se descargaran las imagenes necesarias, ~200-400 MB en total)")

ds = load_dataset("huggan/wikiart", split="train",
                  trust_remote_code=True, streaming=True)

style_col  = "style"
artist_col = "artist"
genre_col  = "genre" if "genre" in ds.features else None

style_names  = ds.features[style_col].names
artist_names = ds.features[artist_col].names
genre_names  = ds.features[genre_col].names if genre_col else []

print(f"Columnas : {list(ds.features.keys())}")
print(f"\nEstilos en el dataset (+ = seleccionados):")
for name in style_names:
    mark = "+" if name in STYLES else " "
    print(f"  [{mark}] {name}")

---
## 3 — Recopilacion, recorte y guardado

Itera en streaming con shuffle. Para en cuanto recoge `N_PER_STYLE` imagenes
de cada estilo. Solo se descargan las imagenes que se van a usar.

In [ ]:
# Recopilación de imágenes en streaming y guardado en disco.
# center_crop_resize aplica un recorte central cuadrado antes de redimensionar
# a 512×512, preservando el centro compositivo de la pintura y evitando
# distorsión de proporciones. Las captions siguen la plantilla
# 'a {adj} {genre} painting by {apellido}' para mantener consistencia con
# el vocabulario de LAION-5B usado en el preentrenamiento de los modelos.
from PIL import Image

def make_caption(style, genre, artist):
    adj  = STYLE_ADJ.get(style, style.lower().replace("_", " "))
    last = artist.split()[-1] if artist else ""
    g    = genre.lower().replace("_", " ") if genre else ""
    if g and last:
        return f"a {adj} {g} painting by {last}"
    if last:
        return f"a {adj} painting by {last}"
    return f"a {adj} painting"

def center_crop_resize(img, size):
    w, h = img.size
    s    = min(w, h)
    img  = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    return img.resize((size, size), Image.LANCZOS)

ds_shuf   = ds.shuffle(seed=RANDOM_SEED, buffer_size=BUFFER_SIZE)
collected = {s: [] for s in STYLES}
needed    = set(STYLES)

print(f"Recopilando {N_PER_STYLE} imagenes por estilo...")
for sample in ds_shuf:
    name = style_names[sample[style_col]]
    if name in needed:
        collected[name].append(sample)
        if len(collected[name]) >= N_PER_STYLE:
            needed.discard(name)
            print(f"  {name:<30} {N_PER_STYLE} recopiladas")
    if not needed:
        break

if needed:
    print(f"\nAVISO: estilos sin suficientes imagenes: {needed}")

# Guardar
manifest   = []
global_idx = 0

for style in STYLES:
    for sample in collected.get(style, []):
        img = sample["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        img        = img.convert("RGB")
        img        = center_crop_resize(img, IMG_SIZE)
        artist_str = artist_names[sample[artist_col]] if artist_names else ""
        genre_str  = genre_names[sample[genre_col]]   if (genre_names and genre_col) else ""
        caption    = make_caption(style, genre_str, artist_str)
        fname      = f"{global_idx:05d}.png"
        img.save(os.path.join(IMAGES_DIR, fname))
        manifest.append({
            "id": global_idx, "filename": fname,
            "style": style, "genre": genre_str,
            "artist": artist_str, "caption": caption,
        })
        global_idx += 1

print(f"\nTotal guardado: {len(manifest)} imagenes en {IMAGES_DIR}")

---
## 4 — Guardado del manifest

In [ ]:
manifest_path = os.path.join(FINETUNE_DIR, "manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

total_mb = sum(
    os.path.getsize(os.path.join(IMAGES_DIR, e["filename"]))
    for e in manifest
) / 1024**2

print(f"manifest.json : {len(manifest)} entradas")
print(f"Tamano total  : {total_mb:.0f} MB")
print(f"Ruta          : {manifest_path}")

---
## 5 — Verificacion

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image

style_counts = Counter(e["style"] for e in manifest)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(list(style_counts.keys()), list(style_counts.values()),
        color="#4C72B0", alpha=0.8)
ax.set_xlabel("No imagenes")
ax.set_title("Distribucion por estilo (WikiArt subset)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FINETUNE_DIR, "dataset_dist.png"), dpi=100)
plt.show()

fig2, axs = plt.subplots(3, 3, figsize=(9, 9))
for ax, entry in zip(axs.flatten(), manifest[:9]):
    img = Image.open(os.path.join(IMAGES_DIR, entry["filename"]))
    ax.imshow(img); ax.axis("off")
    ax.set_title(STYLE_ADJ.get(entry["style"], entry["style"]), fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FINETUNE_DIR, "dataset_preview.png"), dpi=100)
plt.show()

print("Ejemplos de captions:")
for e in manifest[:6]:
    print(f"  [{e['style'][:20]:<20}] {e['caption']}")